In [1]:
from ccka.models.kernel import KernelModel
from ccka.circuits.angleEmbeddingKernel import quackEmbeddingCircuit
from ccka.aligner.kta import centroidBasedKTA
import pennylane as qml
import jax
import jax.numpy as jnp

In [2]:
data = jnp.load('../data/corners.npy', allow_pickle=True).item()
X = jnp.asarray(data['x_train'])
y = jnp.asarray(data['y_train'])
x_test = jnp.asarray(data['x_test'])
y_test = jnp.asarray(data['y_test'])

X_combined = jnp.concatenate([X, x_test], axis=0)
y_combined = jnp.concatenate([y, y_test], axis=0)

N, D = X_combined.shape
(N, D)

(200, 2)

In [3]:
kernel = quackEmbeddingCircuit(
                                num_qubits = 5,
                                reps = 6,
                                reupload = True
                        )
init_weights = kernel.init_weights()
model = KernelModel(circuit = kernel)

In [4]:
# Experiment Configutations

centroids = [2, 4, 6, 8, 10]
num_iterations = [20, 50, 100, 200, 500]


In [5]:
acc = []
alpha = [0.001, 0.01, 0.1, 0.2, 0.5]
for i in range(1):

    kernel = quackEmbeddingCircuit(
                                num_qubits = 4,
                                reps = 5,
                                reupload = True
                        )
    init_weights = kernel.init_weights()
    model = KernelModel(circuit = kernel)

    aligner = centroidBasedKTA(
                    kernel_model= model,
                    data = X_combined,
                    labels = y_combined,
                    matrix_type='regular', #
                    clustering='regular',
                    split_size=0.50,
                    centroids= 4,
                    landmark_points=2,
                    lambda_co=0.0,
                    lambda_kao=0.0,
                    epochs=10,
                    eps=0.001,
                    alpha=0.01
    )

    history = aligner.align()
    acc.append(history['final_svm_metrics']['test_accuracy'])

[CentroidBasedKTA] KTA alignment: 100%|██████████| 10/10 [00:06<00:00,  1.57it/s]

┌────────────────────────────────────────────────────────────────────────────┐
│                              TRAINING SUMMARY                              │
├────────────────────────────────────────────────────────────────────────────┤
│ 'Epochs run          : 10'                                                 │
│ 'Total training time : 6.48 s'                                             │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│                              ACCURACY METRICS                              │
├────────────────────────────────────────────────────────────────────────────┤
│ 'Initial train accuracy : 0.9000'                                          │
│ 'Final   train accuracy : 0.9700'                                          │
│ 'Initial test  accuracy : 0.8500'                                          │
│ 'Final   test  accuracy : 0.9300'                

In [6]:
model.circuit_executions

349240

In [7]:
aligner.get_circuit_executions()

0